# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    


In [3]:
OLLAMA_BASE_URL = "http://192.168.2.2:11434/v1"

openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

MODEL = 'gemma3:12b'
#openai = OpenAI()

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemma3:12b
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company news',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma3:12b
Found 15 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'brand information', 'url': 'https://huggingface.co/brand'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'github repository', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin p

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemma3:12b
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5
Updated
4 days ago
•
168k
•
1.28k
MiniMaxAI/MiniMax-M2.5
Updated
1 day ago
•
31.6k
•
686
Qwen/Qwen3.5-397B-A17B
Updated
1 day ago
•
19.6k
•
553
Nanbeige/Nanbeige4.1-3B
Updated
about 5 hours ago
•
32k
•
523
openbmb/MiniCPM-SALA
Updated
6 days ago
•
3.86k
•
456
Browse 2M+ models
Spaces
Running
755
Demo Playground
⚡
755
Free platform to access multiple AI models
Running
Reachy
294
Reachy Phone Home
📱
294
Phone focus companion for Reachy Mini
Running
Featured
4.73k
Wan2.2 Animate
👁
4.73k
Wan2.2 Animate
Running
on
A100
Featu

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [27]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
Respond in both English and Finnish.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma3:12b
Found 14 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5\nUpdated\n4 days ago\n•\n168k\n•\n1.28k\nMiniMaxAI/MiniMax-M2.5\nUpdated\n1 day ago\n•\n31.6k\n•\n686\nQwen/Qwen3.5-397B-A17B\nUpdated\n1 day ago\n•\n19.6k\n•\n553\nNanbeige/Nanbeige4.1-3B\nUpdated\nabout 5 hours ago\n•\n32k\n•\n523\nopenbmb/MiniCPM-SALA\nUpdated\n6 days ago\n•\n3.86k\n•\n456\nBrowse 2M+ models\nSpaces\nRunning\n755\nDemo Playground\n⚡\n755\nFree platform to a

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemma3:12b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
#create_brochure("HuggingFace", "https://huggingface.co")
create_brochure("Monad", "https://monad.fi")
#create_brochure("rk", "https://rkelkka.fi")

Selecting relevant links for https://monad.fi by calling gemma3:12b
Found 9 relevant links


## Monad: Building Sustainable Software Solutions

**(Image: A modern, clean graphic representing growth or stability - perhaps a stylized tree or a network connection)**

**Who We Are:**

Monad is a Finnish software house based in Tampere, specializing in creating robust, future-proof digital solutions. Founded in 2017, we blend deep technical expertise with a commitment to quality and a genuine passion for our work.  We’re more than just software developers; we're partners dedicated to building lasting value for our clients.

**What We Do:**

We offer a comprehensive suite of services, including:

*   **Data & Analytics:** Leverage the power of data to drive informed decisions.
*   **Smart Technologies:** Embrace cutting-edge innovation for impactful results.
*   **Artificial Intelligence:**  Implement AI solutions to optimize processes and unlock new possibilities.
*   **Service Design:**  Focusing the user in design decisions.
*   **UX/UI Design:** Create intuitive and engaging user experiences.
*   **Cloud Solutions:** Build and deploy scalable and reliable cloud-based applications.
*   **Software Development:** Craft custom software solutions tailored to your specific needs.
*   **DevOps:** Streamline development and deployment processes for faster delivery.

**Our Impact:**

We’ve had the privilege of collaborating with leading organizations across diverse sectors, including:

*   **Energy:** Contributing to critical systems reliability.
*   **Aerospace:** Delivering solutions for demanding industries.
*   **Heavy Machinery:**  Powering innovation in equipment control.
*   **Social and Healthcare:**  Advancing patient care and efficiency.
*   **Industrial:**  Driving automation and optimization.

**(Short testimonial quote pulled from landing page: "Yhteistyö Monadin kanssa toimi loistavasti. Tämä tuntui aidolta kumppanuudelta" – Janne Leinonen, Pirha)**



**Our Culture – Where People Thrive:**

At Monad, we believe in a culture of respect, openness, and continuous growth. We value:

*   **Integrity & Honesty:** Transparency is at the core of our relationships, building trust with clients and colleagues.
*   **Growth-Mindset:** We are constantly learning and evolving, both individually and as a team, staying at the forefront of technology.
*   **Flexibility:** We empower our team to choose their work style, hours, and location, fostering a supportive environment for everyone to be their best.
*   **Relaxation:** Balancing dedication and precision with a relaxed atmosphere.

We champion equality, encouraging all individuals to lead and contribute their ideas.  Our team members are curious, responsible, and proud of their profession. We celebrate individuality and mutual respect.



**Join Our Team:**

Are you a passionate and skilled software professional?  We're always looking for talented individuals to join our growing team! We offer a challenging and rewarding work environment where you can make a real impact.  We want people who are driven, collaborative, and eager to learn. Come and build a smarter future with us!

**(Link to careers page)**



**Let's Build Something Amazing Together:**

**[Monad Website Link]**
**[LinkedIn Link]**
**[Facebook Link]**
**[Twitter Link]**
**[Instagram Link]**
**[Github Link]**



**(Contact Information)**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemma3:12b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
#stream_brochure("HuggingFace", "https://huggingface.co")
stream_brochure("Monad", "https://monad.fi")


Selecting relevant links for https://monad.fi by calling gemma3:12b
Found 10 relevant links


## Monad - Sustainable Software Development

**(Brochure - English)**

**About Monad**

Monad is a software development company based in Tampere, Finland, specializing in building robust and sustainable digital solutions. Established in 2017, we're driven by a passion for high-quality software and a commitment to creating solutions that are user-friendly, developer-friendly, and built to last. 

**What We Do**

We work with clients across a diverse range of industries, including:

*   Aerospace
*   Energy
*   Industrial Machinery
*   Social and Healthcare
*   Industry

Our services include:

*   Data & Analytics
*   Smart Technologies
*   Artificial Intelligence (AI)
*   Service Design 
*   UX/UI Design
*   Cloud Solutions
*   Software Development
*   DevOps

**Our Culture**

At Monad, we believe in creating a workplace where everyone can thrive.  We foster a culture built on:

*   **Respect & Honesty:** Transparency and open communication are key.
*   **Continuous Growth:** We invest in our people and stay at the forefront of technology.
*   **Autonomy & Flexibility:** We empower our team members with options for how, when, and where they work.
*   **Collaboration:** We welcome diverse perspectives and strive to learn from one another and our partners.  We treat every employee as a valued individual.

**Why Choose Monad?**

*   **Sustainable Solutions:** We build digital solutions that endure – in value and functionality.
*   **Expertise:**  Years of experience in critical software environments.
*   **Partnership:** We are committed to building strong, collaborative relationships with our clients.


**Join Our Team!**

We are always seeking talented and passionate individuals to join our growing team. If you're looking for a company that values its people and encourages innovation, we invite you to explore our open positions! [Link to Careers Page]


## Monad - Kestävää ohjelmistokehitystä

**(Brochure - Finnish)**

**Tietoa Monadista**

Monad on tamperelainen ohjelmistotalo, joka on rakentanut digitaalisia ratkaisuja ammattitaidolla vuodesta 2017. Olemme erikoistuneet kestävien ja laadukkaiden digitaalisten ratkaisujen kehittämiseen, jotka ovat käyttäjäystävällisiä, kehittäjäystävällisiä ja suunniteltu kestämään aikaa.

**Mitä teemme**

Palvelemme asiakkaita monipuolisilla toimialoilla, kuten:

*   Ilmailu
*   Energia
*   Teollisuus
*   Sosiaali- ja terveysala
*   Työkoneala

Palveluihimme kuuluvat:

*   Data & Analytiikka
*   Älykkäät teknologiat
*   Tekoäly (AI)
*   Palvelumuotoilu
*   UX/UI-design
*   Pilviratkaisut
*   Ohjelmistokehitys
*   DevOps

**Yrityskulttuurimme**

Monadilla uskomme, että parhaat tulokset syntyvät, kun jokainen voi vaikuttaa ja kehittyä. Kulttuurimme perustuu:

*   **Arvostukseen ja rehellisyyteen:** Läpinäkyvyys on avain luottamukseen.
*   **Jatkuvaan kehitykseen:** Investoimme osaamiseen ja pysymme teknologian kärjessä.
*   **Joustavuuteen:** Työntekijät voivat valita työtapansa, työaikansa ja paikkansa.
*   **Yhteistyöhön:** Arvostamme erilaisia näkökulmia ja pyrimme oppimaan toisiltamme ja kumppaneiltamme. Kohtelemme jokaista työntekijää arvokkaana yksilönä.

**Miksi valita Monad?**

*   **Kestävät ratkaisut:** Rakennamme digitaalisia ratkaisuja, jotka kestävät aikaa – sekä arvoltaan että toimivuudeltaan.
*   **Asiantuntemus:** Vuosien kokemus kriittisissä ohjelmistoympäristöissä.
*   **Kumppanuus:** Olemme sitoutuneet rakentamaan vahvoja ja yhteistyöhaluisia asiakassuhteita.

**Liity tiimiimme!**

Etsimme jatkuvasti osaavia ja intohimoisia ihmisiä liittymään kasvavaan tiimiimme. Jos etsit yritystä, joka arvostaa työntekijöitään ja kannustaa innovaatioita, kutsumme sinut tutustumaan avoimiin työpaikkoihimme! [Linkki Urat-sivulle]

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>